# GTA 6 AutoClipper — Universal Colab Production Studio

**One notebook for every production type.** Choose the workflow from the control panel instead of opening or uploading a different notebook.

### Production modes
- **Gaming highlights:** analyse one completed, permitted gameplay video and generate reviewable highlight candidates.
- **AI scene assembly:** arrange AI-generated scene videos into one finished video. Prefix filenames with 01_, 02_, 03_ ... to control their order.
- **Long-form video:** combine one or more permitted video segments into a standard horizontal YouTube video.
- **Multi-clip compilation:** combine several permitted clips into one finished video.

### Reuse instead of uploading again
Sources, intermediate files, settings and exports are saved in Google Drive under MyDrive/GTA6_AutoClipper/projects/. On a later Colab session, select **Reuse this project's saved sources** and enter the same project name.

### Important boundaries
This notebook edits and assembles video files you provide. It does not generate new AI scenes, scrape watch pages, bypass DRM, or download arbitrary Kick/Twitch/YouTube streams. Use footage and audio you own or are licensed/authorized to edit and redistribute. A public stream, a credit line, or an edit does not itself grant those rights. Nothing is published automatically; exports are for your review and manual upload.

### YouTube Shorts format check
For a standard YouTube channel, YouTube currently classifies square or vertical videos up to **3 minutes** as Shorts. This notebook exports vertical 9:16 or horizontal 16:9 and stops the Shorts export when the assembled vertical video exceeds 3 minutes. See the official [three-minute Shorts rules](https://support.google.com/youtube/answer/15424877).

### Monetization reminder
Copyright permission and YouTube monetization eligibility are separate. Reposting clips or assembling scenes with little original narrative may fail the reused-content or repetitive-content review. Add your own meaningful storyline, commentary, explanation or creative contribution; no export setting can guarantee monetization. See [YouTube's channel monetization policies](https://support.google.com/youtube/answer/1311392).

In [ ]:
import os, sys, json, re, time, uuid, shutil, subprocess, zipfile
from pathlib import Path
from google.colab import drive, files
from IPython.display import display, Markdown, Video

drive.mount("/content/drive")

REPO_URL = "https://github.com/topboyasian-stack/gta6-autoclipper.git"
ROOT = Path("/content/gta6-autoclipper")
if not (ROOT / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(ROOT)

if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "fastapi>=0.115,<1", "python-multipart>=0.0.9"],
    check=True,
)
import ipywidgets as widgets

WORKSPACE = Path("/content/drive/MyDrive/GTA6_AutoClipper")
PROJECTS_DIR = WORKSPACE / "projects"
INCOMING_DIR = WORKSPACE / "incoming"
PROJECTS_DIR.mkdir(parents=True, exist_ok=True)
INCOMING_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".webm", ".m4v"}

mode_widget = widgets.Dropdown(
    options=[
        ("Gaming highlights from one completed video", "Gaming highlights"),
        ("Assemble AI-generated scenes", "AI scene assembly"),
        ("Create a long-form YouTube video", "Long-form video"),
        ("Create a multi-clip compilation", "Multi-clip compilation"),
    ], value="AI scene assembly", description="Production:", layout=widgets.Layout(width="95%")
)
format_widget = widgets.Dropdown(
    options=[("YouTube Shorts — vertical 9:16", "shorts"),
             ("Standard YouTube — horizontal 16:9", "standard")],
    value="shorts", description="Format:", layout=widgets.Layout(width="95%")
)
source_widget = widgets.Dropdown(
    options=[("Reuse this project's saved sources", "reuse"),
             ("Use a folder already in Google Drive", "drive"),
             ("Upload file(s) from this device once", "upload")],
    value="upload", description="Sources:", layout=widgets.Layout(width="95%")
)
project_widget = widgets.Text(value="GTA6_First_Production", description="Project:", layout=widgets.Layout(width="95%"))
drive_folder_widget = widgets.Text(value=str(INCOMING_DIR), description="Drive folder:", layout=widgets.Layout(width="95%"))
clip_seconds_widget = widgets.IntSlider(value=35, min=10, max=90, step=5, description="Clip length:", continuous_update=False, layout=widgets.Layout(width="95%"))
candidate_widget = widgets.IntSlider(value=3, min=1, max=8, step=1, description="Candidates:", continuous_update=False, layout=widgets.Layout(width="95%"))
creator_widget = widgets.Text(value="", description="Creator:", placeholder="Optional", layout=widgets.Layout(width="95%"))
source_url_widget = widgets.Text(value="", description="Source URL:", placeholder="Optional", layout=widgets.Layout(width="95%"))
rights_widget = widgets.Textarea(value="", description="Rights basis:", placeholder="Explain why you own or are authorized/licensed to edit and redistribute every source video and its audio.", layout=widgets.Layout(width="95%", height="85px"))
mute_widget = widgets.Checkbox(value=False, description="Mute all original audio")
rights_check_widget = widgets.Checkbox(value=False, description="I confirm I have the necessary rights for editing and redistribution of all selected footage/audio.", indent=False)
download_widget = widgets.Checkbox(value=True, description="Download the finished ZIP to this device")

display(widgets.VBox([
    widgets.HTML("<h3>1. Production settings</h3>"), mode_widget, format_widget, project_widget,
    widgets.HTML("<h3>2. Source selection</h3>"), source_widget, drive_folder_widget,
    widgets.HTML("<h3>3. Editing settings</h3>"), clip_seconds_widget, candidate_widget, mute_widget,
    widgets.HTML("<h3>4. Source details and rights confirmation</h3>"),
    creator_widget, source_url_widget, rights_widget, rights_check_widget, download_widget,
]))
print(f"Persistent Drive workspace: {WORKSPACE}")

In [ ]:
MODE = mode_widget.value
OUTPUT_FORMAT = format_widget.value
SOURCE_MODE = source_widget.value
PROJECT_NAME = project_widget.value.strip()
CLIP_SECONDS = int(clip_seconds_widget.value)
MAX_CANDIDATES = int(candidate_widget.value)
CREATOR = creator_widget.value.strip()
SOURCE_URL = source_url_widget.value.strip()
RIGHTS_BASIS = rights_widget.value.strip()
MUTE_AUDIO = bool(mute_widget.value)
DOWNLOAD_EXPORT = bool(download_widget.value)

if not PROJECT_NAME:
    raise ValueError("Enter a project name.")
if len(RIGHTS_BASIS) < 8:
    raise ValueError("Enter the actual rights basis for the selected footage.")
if not rights_check_widget.value:
    raise PermissionError("Confirm the rights basis only after checking every selected source and its audio.")

safe_project_name = re.sub(r"[^A-Za-z0-9_-]+", "_", PROJECT_NAME).strip("_")[:70] or "video_project"
PROJECT_DIR = PROJECTS_DIR / safe_project_name
SOURCES_DIR, WORK_DIR, EXPORTS_DIR = PROJECT_DIR / "sources", PROJECT_DIR / "work", PROJECT_DIR / "exports"
for folder in (SOURCES_DIR, WORK_DIR, EXPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

natural_key = lambda p: [(0, int(x)) if x.isdigit() else (1, x.lower()) for x in re.split(r"(\d+)", p.name)]
if SOURCE_MODE == "reuse":
    source_files = sorted([p for p in SOURCES_DIR.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS], key=natural_key)
    if not source_files:
        raise FileNotFoundError(f"No saved sources found in {SOURCES_DIR}. First choose a Drive folder or upload files.")
elif SOURCE_MODE == "drive":
    input_folder = Path(drive_folder_widget.value.strip()).expanduser()
    if not input_folder.is_dir():
        raise FileNotFoundError(f"Drive folder not found: {input_folder}")
    incoming = sorted([p for p in input_folder.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS], key=natural_key)
    if not incoming:
        raise FileNotFoundError(f"No supported video files found in {input_folder}")
    source_files = []
    for item in incoming:
        safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", item.name).strip("_") or f"source_{len(source_files)+1}.mp4"
        target = SOURCES_DIR / safe_name
        if item.resolve() != target.resolve():
            shutil.copy2(item, target)
        source_files.append(target)
else:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No files were selected.")
    source_files = []
    for filename, data in uploaded.items():
        if Path(filename).suffix.lower() not in VIDEO_EXTENSIONS:
            print(f"Skipping unsupported file: {filename}")
            continue
        safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", Path(filename).name).strip("_")
        target = SOURCES_DIR / (safe_name or f"upload_{len(source_files)+1}.mp4")
        target.write_bytes(data)
        source_files.append(target)
    source_files.sort(key=natural_key)
    if not source_files:
        raise RuntimeError("No supported video files were uploaded.")

if MODE == "Gaming highlights" and len(source_files) != 1:
    raise ValueError("Gaming highlights mode needs exactly one completed source video.")
if MODE in {"AI scene assembly", "Multi-clip compilation"} and len(source_files) < 2:
    raise ValueError(f"{MODE} needs at least two video files. Long-form mode accepts one or more.")
if MODE == "Long-form video" and OUTPUT_FORMAT != "standard":
    raise ValueError("Long-form video mode requires Standard YouTube (horizontal 16:9). Choose that format before continuing.")

source_fingerprint = "|".join(f"{p.name}:{p.stat().st_size}:{p.stat().st_mtime_ns}" for p in source_files)
state_path = PROJECT_DIR / "project_state.json"
old_state = json.loads(state_path.read_text()) if state_path.exists() else {}
source_id = old_state.get("source_id") if old_state.get("source_fingerprint") == source_fingerprint else uuid.uuid4().hex
PROJECT_STATE = {
    "project_name": PROJECT_NAME, "mode": MODE, "output_format": OUTPUT_FORMAT,
    "source_files": [str(p) for p in source_files], "source_fingerprint": source_fingerprint,
    "source_id": source_id, "creator": CREATOR, "source_url": SOURCE_URL,
    "rights_basis": RIGHTS_BASIS, "clip_seconds": CLIP_SECONDS,
    "max_candidates": MAX_CANDIDATES, "mute_audio": MUTE_AUDIO,
    "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
state_path.write_text(json.dumps(PROJECT_STATE, indent=2), encoding="utf-8")

print(f"Project saved: {PROJECT_DIR}")
print("Processing order:")
for i, path in enumerate(source_files, 1):
    print(f"  {i:02d}. {path.name}")
print("These source files now persist in Google Drive and can be reused without uploading them again.")

In [ ]:
import importlib
from IPython.display import display, Markdown, Video

WIDTH, HEIGHT = (1080, 1920) if OUTPUT_FORMAT == "shorts" else (1920, 1080)
candidate_records, approval_checks = [], []
draft_video_path = None

def run_checked(command, label):
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"{label} failed:\n{(result.stderr or result.stdout or '')[-3000:]}")
    return result

if MODE == "Gaming highlights":
    os.environ["DATA_DIR"] = str(PROJECT_DIR / "clipper_data")
    os.environ["CLIP_SECONDS"] = str(CLIP_SECONDS)
    os.environ["MAX_CANDIDATES"] = str(MAX_CANDIDATES)
    os.environ["MUTE_SOURCE_AUDIO"] = "1" if MUTE_AUDIO else "0"
    sys.path.insert(0, str(ROOT))
    import app.main as clipper
    clipper = importlib.reload(clipper)
    clipper.CLIP_SECONDS, clipper.MAX_CANDIDATES = CLIP_SECONDS, MAX_CANDIDATES

    # Use the existing highlight detector, but let it render either aspect ratio.
    def render_for_selected_format(source, target, start, length):
        vf = (
            f"[0:v]split=2[bg][fg];"
            f"[bg]scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=increase,crop={WIDTH}:{HEIGHT},boxblur=20:10[bg];"
            f"[fg]scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=decrease[fg];"
            f"[bg][fg]overlay=(W-w)/2:(H-h)/2,setsar=1[v]"
        )
        cmd = ["ffmpeg", "-y", "-ss", f"{start:.2f}", "-i", str(source), "-t", f"{length:.2f}",
               "-filter_complex", vf, "-map", "[v]", "-c:v", "libx264", "-preset", "veryfast",
               "-crf", "22", "-pix_fmt", "yuv420p"]
        cmd += ["-an"] if MUTE_AUDIO else ["-map", "0:a?", "-c:a", "aac", "-b:a", "160k"]
        cmd += ["-movflags", "+faststart", str(target)]
        run_checked(cmd, f"Rendering candidate {target.name}")

    clipper.render_clip = render_for_selected_format
    source_path, source_id = source_files[0], PROJECT_STATE["source_id"]
    with clipper.db() as con:
        con.execute(
            "INSERT OR IGNORE INTO sources(id,filename,path,creator,source_url,rights_basis) VALUES(?,?,?,?,?,?)",
            (source_id, source_path.name, str(source_path), CREATOR, SOURCE_URL, RIGHTS_BASIS),
        )
        existing = con.execute("SELECT COUNT(*) n FROM clips WHERE source_id=?", (source_id,)).fetchone()["n"]
    if not existing:
        result = clipper.create_candidates(source_id, source_path)
        print(f"Created {result['candidates_created']} candidate(s).")
    else:
        print("Reusing saved candidates from this project.")

    with clipper.db() as con:
        rows = con.execute(
            "SELECT c.*, s.filename, s.creator, s.source_url, s.rights_basis "
            "FROM clips c JOIN sources s ON s.id=c.source_id WHERE c.source_id=? ORDER BY c.start",
            (source_id,),
        ).fetchall()
    if not rows:
        raise RuntimeError("No highlight candidates are available.")
    display(Markdown("### Review the candidates. Previously approved selections remain checked."))
    for index, row in enumerate(rows, 1):
        item = dict(row)
        candidate_records.append(item)
        display(Markdown(f"#### Candidate {index} — start {int(row['start'])}s — {row['status']}"))
        display(Video(filename=str(row["path"]), embed=True, width=320))
        check = widgets.Checkbox(value=row["status"] == "approved",
                                 description=f"Include candidate {index}",
                                 disabled=row["status"] == "rejected")
        approval_checks.append(check)
        display(check)

else:
    # AI scenes, long-form video and compilations share the same assembly engine.
    normalized_dir = WORK_DIR / f"normalized_{WIDTH}x{HEIGHT}_{'mute' if MUTE_AUDIO else 'audio'}"
    normalized_dir.mkdir(parents=True, exist_ok=True)
    normalized_files = []

    def has_audio(path):
        result = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "a:0", "-show_entries", "stream=index", "-of", "csv=p=0", str(path)],
            capture_output=True, text=True,
        )
        return result.returncode == 0 and bool(result.stdout.strip())

    vf = (
        f"[0:v]split=2[bg][fg];"
        f"[bg]scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=increase,crop={WIDTH}:{HEIGHT},boxblur=20:10[bg];"
        f"[fg]scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=decrease[fg];"
        f"[bg][fg]overlay=(W-w)/2:(H-h)/2,setsar=1,fps=30,format=yuv420p[v]"
    )

    for index, source in enumerate(source_files, 1):
        stat = source.stat()
        safe_stem = re.sub(r"[^A-Za-z0-9_-]+", "_", source.stem).strip("_") or "scene"
        normalized = normalized_dir / f"{index:03d}_{safe_stem}_{stat.st_size}_{stat.st_mtime_ns}.mp4"
        if not normalized.exists():
            audio_present = has_audio(source) and not MUTE_AUDIO
            cmd = ["ffmpeg", "-y", "-i", str(source)]
            if not MUTE_AUDIO and not audio_present:
                cmd += ["-f", "lavfi", "-i", "anullsrc=channel_layout=stereo:sample_rate=48000"]
            graph = vf
            if audio_present:
                graph += ";[0:a:0]aresample=48000,aformat=sample_fmts=fltp:channel_layouts=stereo,apad[a]"
            cmd += ["-filter_complex", graph, "-map", "[v]"]
            if MUTE_AUDIO:
                cmd += ["-an"]
            elif audio_present:
                cmd += ["-map", "[a]", "-c:a", "aac", "-ar", "48000", "-ac", "2", "-b:a", "160k", "-shortest"]
            else:
                cmd += ["-map", "1:a:0", "-c:a", "aac", "-ar", "48000", "-ac", "2", "-b:a", "160k", "-shortest"]
            cmd += ["-c:v", "libx264", "-preset", "veryfast", "-crf", "22", "-pix_fmt", "yuv420p",
                    "-r", "30", "-movflags", "+faststart", str(normalized)]
            run_checked(cmd, f"Preparing {source.name}")
        normalized_files.append(normalized)
        print(f"Prepared {index}/{len(source_files)}: {source.name}")

    concat_list = WORK_DIR / "concat_list.txt"
    def concat_escape(path):
        return str(path.resolve()).replace("\\", "/").replace("'", "'\\''")
    concat_list.write_text("".join(f"file '{concat_escape(p)}'\n" for p in normalized_files), encoding="utf-8")
    label = {"AI scene assembly": "ai_scenes", "Long-form video": "long_form", "Multi-clip compilation": "compilation"}[MODE]
    draft_video_path = WORK_DIR / f"{safe_project_name}_{label}_{WIDTH}x{HEIGHT}_draft.mp4"
    draft_video_path.unlink(missing_ok=True)
    run_checked(
        ["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(concat_list), "-c", "copy", "-movflags", "+faststart", str(draft_video_path)],
        "Joining scenes",
    )
    if OUTPUT_FORMAT == "shorts":
        duration_result = run_checked(
            ["ffprobe", "-v", "error", "-show_entries", "format=duration",
             "-of", "default=noprint_wrappers=1:nokey=1", str(draft_video_path)],
            "Checking Shorts duration",
        )
        duration_seconds = float(duration_result.stdout.strip())
        if duration_seconds > 180:
            raise ValueError(
                f"This vertical draft is {duration_seconds/60:.1f} minutes. "
                "YouTube Shorts must be square or vertical and no longer than 3 minutes. "
                "Choose Standard YouTube (16:9) or shorten the selected scenes, then rerun."
            )
    print(f"Draft assembled: {draft_video_path}")
    print("Watch the draft before exporting.")
    display(Video(filename=str(draft_video_path), embed=True, width=480))

In [ ]:
stamp = time.strftime("%Y%m%d_%H%M%S")
aspect = "9x16" if OUTPUT_FORMAT == "shorts" else "16x9"
stem = re.sub(r"[^A-Za-z0-9_-]+", "_", PROJECT_NAME).strip("_")[:60]
zip_path = EXPORTS_DIR / f"{stem}_{stamp}.zip"
manifest = {
    "project": PROJECT_NAME, "production_mode": MODE, "format": OUTPUT_FORMAT,
    "resolution": f"{WIDTH}x{HEIGHT}", "aspect_ratio": aspect,
    "creator": CREATOR, "source_url": SOURCE_URL, "rights_basis_recorded": bool(RIGHTS_BASIS),
    "source_files": [p.name for p in source_files], "mute_original_audio": MUTE_AUDIO,
    "exported_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

if MODE == "Gaming highlights":
    approved = []
    with clipper.db() as con:
        for item, check in zip(candidate_records, approval_checks):
            status = "approved" if check.value else "rejected"
            con.execute("UPDATE clips SET status=? WHERE id=? AND status IN ('pending','approved')", (status, item["id"]))
            if check.value:
                approved.append(item)
    if not approved:
        raise RuntimeError("No candidates were approved. Tick at least one candidate.")
    title = f"{PROJECT_NAME} gaming highlight" + (" #Shorts" if OUTPUT_FORMAT == "shorts" else "")
    metadata = [
        f"Suggested title: {title}", "", "Suggested description:",
        "Gaming highlight edit from the supplied source footage.",
        f"Original creator: {CREATOR or 'Not supplied'}",
        f"Rights basis recorded: {RIGHTS_BASIS}",
        "Review video/audio rights before publishing. Attribution is not a substitute for permission.",
    ]
    if SOURCE_URL.startswith(("https://", "http://")):
        metadata.insert(5, f"Original source: {SOURCE_URL}")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        approved_manifest = []
        for number, item in enumerate(approved, 1):
            clip = Path(item["path"])
            name = f"{number:02d}_{clip.name}"
            archive.write(clip, arcname=name)
            approved_manifest.append({"file": name, "clip_id": item["id"], "start_seconds": item["start"], "duration_seconds": item["duration"]})
        manifest["approved_clips"] = approved_manifest
        archive.writestr("approval_manifest.json", json.dumps(manifest, indent=2))
        archive.writestr("youtube_metadata.txt", "\n".join(metadata) + "\n")
else:
    if draft_video_path is None or not draft_video_path.is_file():
        raise FileNotFoundError("Draft video missing. Run the processing cell first.")
    decision = input("After watching the draft, type EXPORT to package it: ").strip()
    if decision != "EXPORT":
        raise RuntimeError("Export cancelled. Your draft and sources remain saved in Drive.")
    final_video = EXPORTS_DIR / f"{stem}_{aspect}_{stamp}.mp4"
    shutil.copy2(draft_video_path, final_video)
    title = f"{PROJECT_NAME} #Shorts" if OUTPUT_FORMAT == "shorts" else f"{PROJECT_NAME} — Full Video"
    metadata = [
        f"Suggested title: {title}", "", "Suggested description:",
        f"Production type: {MODE}.",
        "This video was assembled from the source video files saved in this project.",
        f"Rights basis recorded: {RIGHTS_BASIS}",
        "Review video/audio rights before publishing. Attribution is not a substitute for permission.",
    ]
    if CREATOR:
        metadata.append(f"Creator: {CREATOR}")
    if SOURCE_URL.startswith(("https://", "http://")):
        metadata.append(f"Source: {SOURCE_URL}")
    metadata_path = EXPORTS_DIR / f"{stem}_{stamp}_youtube_metadata.txt"
    metadata_path.write_text("\n".join(metadata) + "\n", encoding="utf-8")
    manifest["final_video"] = final_video.name
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(final_video, arcname=final_video.name)
        archive.write(metadata_path, arcname=metadata_path.name)
        archive.writestr("project_manifest.json", json.dumps(manifest, indent=2))

print(f"Export ZIP saved in Google Drive: {zip_path}")
print(f"Project folder: {PROJECT_DIR}")
if DOWNLOAD_EXPORT:
    files.download(str(zip_path))